# Aula Lab. 2 - OpenCV
## Amostragem, Quantização e Representação
Nestes exercícios vamos entnder os conceitos de:
* Amostragem, ou Resolução Espacial; a quantidade horizontal, vertical e total de _pixels_ da imagem
* Quantização, ou Profundaidade de Cores; a quantidade de bits para representar uma intensidade luninosa ou de cor do _pixel_
* Representação; o padrão utilizado para representação, a quantidade de canais e o formato do _pixel_ em memória

Começamos com o cabeçaçho padrão:

In [26]:
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import math

cv.samples.addSamplesDataSearchPath("./data")

## 1. Amostragem (física)
Amostragem ou resolução diz respeito à quantidade de amostras que tiramos por unidade de área. Exemplo: Uma foto de satélite para topologia pode ter a resoulção de 1 pixel por m<sup>2</sup>, assim uma imagem de 4.000x4.000 pixels representa uma área de 16Km<sup>2</sup>.

Para a captura de imagens, o sensor fornece a quantidade bruta de pixels dispostos na área de captura, ex.:
* Sony IMX253 possui 12Mpixels distribuidos em uma área aprox. de 1.1". As medidas do fabricante: 14.2mm x 10.4mm, o que resulta em 4096x3000 pixels de resolução.

As lentes atuam como escala, para comprimir uma área maior dentro da áera menor.

Para a apresentação de imagens, a tela fornece a quantidade bruta de pixels dispostos na área de apresentação, ex.:
* Monitor 4K (3840x2160 pixels) de 32" possui uma densidade de aprox. 138 ppi ou dpi (points per inch/dots per inch).
* Celular 4K (3840x2160 pixels) de 6.5" possui uma densidade de aprox. 643 ppi ou dpi (points per inch/dots per inch).

### Exercício - Diminuindo a resolução (ou qtd. de amostras)
Você pode dimiuir a resolução de uma imagem, pelo processo de descarte de amostras. 

1. Carregue a imagem Baboon.jpg em tons de cinza, imprima sua resolução (deve ser 512x512)
1. Crie uma cópia da imagem usando apenas as linhas pares na horizontal e vertical (256x256)
1. Crie uma nova cópia da imagem usando apenas as linhas pares na horizontal e vertical (128x128)
1. Crie uma nova cópia da imagem usando apenas as linhas pares na horizontal e vertical (64x64)
1. Mostre-as em uma linha (subplot 1x4)

> Novidade, Numpy permite que o operador de range de arrays possua 3 valores Início:Fim:Passo, para selecionar todas as linhas e colunas pares: [::2, ::2]

O Matplot irá deixar cada subplot com o mesmo tamanho físico na sua tela, ou seja, o tamanho do pixel ficará maior em cada iteração.

Para cada iteração, perceba que uma vez que a imagem tenha sido reamostrada para uma resolução menor, houve perda de informação (jogamos fora 3 pixels a cada quadrado 2x2).

<table>
  <tr><td>22&check;</td><td>56&cross;</td><td>75&check;</td><td>31&cross;</td></tr>
  <tr><td>16&cross;</td><td>46&cross;</td><td>38&cross;</td><td>20&cross;</td></tr>
  <tr><td>18&check;</td><td>43&cross;</td><td>44&check;</td><td>18&cross;</td></tr>
  <tr><td>24&cross;</td><td>40&cross;</td><td>62&cross;</td><td>23&cross;</td></tr>
</table>

Para deixar com o mesmo tamanho físico, o Matplot irá aumentar (x2, x4, x8) o tamanho da figura, duplicando o valor do pixel para a direita, abaixo e diagonal a cada iteração.

<table>
  <tr><td>22</td><td>22</td><td>75</td><td>75</td></tr>
  <tr><td>22</td><td>22</td><td>75</td><td>75</td></tr>
  <tr><td>18</td><td>18</td><td>44</td><td>44</td></tr>
  <tr><td>18</td><td>18</td><td>44</td><td>44</td></tr>
</table>


In [ ]:
baboon = cv.imread(cv.samples.findFile("baboon.jpg"), cv.IMREAD_GRAYSCALE)
print(f"Resolução original: {baboon.shape}")

baboon_256 = baboon[::2, ::2]
baboon_128 = baboon_256[::2, ::2]
baboon_64  = baboon_128[::2, ::2]

fig, axes = plt.subplots(1, 4)
for ax, img_s, title in zip(axes, [baboon, baboon_256, baboon_128, baboon_64], [512, 256, 128, 64]):
    ax.imshow(img_s, cmap='gray')
    ax.set_title(str(title))
    ax.axis('off')
plt.show()

### Exercício - Desenhando uma circunferência

* Vamos começar com uma imagem (matriz) quadrada (MxM) de 64 pixels.
> np.zeros((h,v), dtype=np.uint8) cria uma matriz preenchida com zeros

In [ ]:
M = 64
img = np.zeros((M, M), dtype=np.uint8)

* Vamos precisar do centro e do raio.
    * O Raio da circunferência será metade da resoloução (M/2).
* Vamos centralizar o círculo na imagem, então o centro também estará na metade da resoloução (M/2).

**Problema**

O cos/sen nos retorna a posição x/y entre -1 até 1 inclusos, ou seja, ao multiplicar pelo raio teremos -M/2 até M/2.

Devemos ajustar para início em 0, basta somar o centro (M/2), ficando de 0 até **M**. Mas a matriz vai de 0 até **M-1**!

* Se M for ímpar: centro (M/2, M/2)
* Se M for par: centro (M/2-1, M/2-1)
    *  Isso fará com que o menor valor seja -1. Mas em Python, índice -1 é o mesmo que o índice máximo, ou seja M/2-1, e não causará erro

> lembre-se, em Python use '//' para divisão inteira

In [ ]:
R = M // 2
Cx = M // 2 - 1
Cy = M // 2 - 1
print(f"M={M}, Cx={Cx}, Cy={Cy}, R={R}")

Uma circunferência inteira possui 360graus (2&pi;) ou seja, um loop entre 0 até 359.
Quantas amostras de ângulos iremos precisar?
Nyquist diz que precisamos do dobro de amostras, como estamos trabalhando em 2D, o dobro do dobro da resolução
>  np.linspace(start, stop, divisions) cria uma lista com a qtd. de valores igualmente espaçados 

In [ ]:
n_samples = 4 * M * M
for angle in np.linspace(0, 360, n_samples):
    rad = np.deg2rad(angle)
    x = int(round(np.cos(rad) * R)) + Cx
    y = int(round(np.sin(rad) * R)) + Cy
    img[y, x] = 255

plt.imshow(img, cmap='gray')
plt.axis('off')
plt.show()

Altere os valores de M para 128, 256 e 512 para validar, quando maior a resolução mais definido ficará o circulo.

Use o espaço de código abaixo para colocar sua implementação e plot das 3 variações (subplot 1x3)
> Não se esqeça, ao fazer alterações rodar os 3 trechos de código.

In [ ]:
resolutions = [128, 256, 512]
images = []
for res in resolutions:
    M = res
    img_c = np.zeros((M, M), dtype=np.uint8)
    R = M // 2
    Cx = M // 2 - 1
    Cy = M // 2 - 1
    n_samples = 4 * M * M
    for angle in np.linspace(0, 360, n_samples):
        rad = np.deg2rad(angle)
        x = int(round(np.cos(rad) * R)) + Cx
        y = int(round(np.sin(rad) * R)) + Cy
        img_c[y, x] = 255
    images.append((img_c, res))

fig, axes = plt.subplots(1, 3)
for ax, (im, res) in zip(axes, images):
    ax.imshow(im, cmap='gray')
    ax.set_title(str(res))
    ax.axis('off')
plt.show()

### Exercício - Quantização/Profundidade de Pixels

Vamos usar como exemplo a imagem squirrel_cls.jpg, carregada colorida RGB.

Nossa imagem de exemplo já possue 8bits de resolução de profundaidade por canal (8R8G8B). Então vamos testar com valores de 4bpp (bits per plane).

A quantização consistem em uma operação de escala, ou seja, multiplicar os valores de entrada por uma escala para obter o valor de saída.

Queremos passar de 8bits para 4bits, nossa escala é passar de 256 (2<sup>8</sup>) para 16 (2<sup>4</sup>) então nossa escala é 1/16, ous seja, basta dividir cada componente do pixel por 16.
<table>
    <tr><td colspan="8">236</td></tr>
    <tr><td>128</td><td>64</td><td>32</td><td>16</td><td>8</td><td>4</td><td>2</td><td>1</td></tr>
    <tr><td>1</td><td>1</td><td>1</td><td>0</td><td>1</td><td>1</td><td>0</td><td>0</td></tr>
</table>

Isso irá resultar em uma imagem escura.
<table>
    <tr><td colspan="8">14 (236/16)</td></tr>
    <tr><td>128</td><td>64</td><td>32</td><td>16</td><td>8</td><td>4</td><td>2</td><td>1</td></tr>
    <tr><td>0</td><td>0</td><td>0</td><td>0</td><td>1</td><td>1</td><td>1</td><td>0</td></tr>
</table>

Podemos reescalar nossos valores novamente para 8bits, multiplicando por 16. Com isso a imagem ficará mais clara.
<table>
    <tr><td colspan="8">224 (14*16)</td></tr>
    <tr><td>128</td><td>64</td><td>32</td><td>16</td><td>8</td><td>4</td><td>2</td><td>1</td></tr>
    <tr><td>1</td><td>1</td><td>1</td><td>0</td><td>0</td><td>0</td><td>0</td><td>0</td></tr>
</table>

**Resultado esperado**: praticamente a mesma imagem, mas com o efeito "banding" que são anéis de cores nas regiões de degradê.

Crie seu código, com subplot, para mostrar a imagem original, a imagem escalada (escura) e a imagem escalada (clara).

> lembre-se: Numpy/Python converte os tipos para float quando faz multiplicação/divisão, deve-se usar .astype como no exemplo da aula1 para voltar para int de 8bits

> Use plt.gcf().set_dpi(X) para aumentar o tamanho da imagem e visualizar melhor, use valores como 10, 150, 200, 250 ou 300 

In [ ]:
img_8bpp = cv.imread(cv.samples.findFile("squirrel_cls.jpg"))
img_8bpp = cv.cvtColor(img_8bpp, cv.COLOR_BGR2RGB)

img_4bpp_dark  = (img_8bpp / 16).astype(np.uint8)
img_4bpp_clear = (img_4bpp_dark * 16).astype(np.uint8)

ax = plt.subplot(1, 3, 1)
ax.imshow(img_8bpp); ax.axis('off'); ax.set_title('8bpp (original)')

ax = plt.subplot(1, 3, 2)
ax.imshow(img_4bpp_dark); ax.axis('off'); ax.set_title('4bpp escura')

ax = plt.subplot(1, 3, 3)
ax.imshow(img_4bpp_clear); ax.axis('off'); ax.set_title('4bpp clara (reescalada)')

plt.gcf().set_dpi(150)
plt.show()

**Máscara de bits**

Existem uma técnica rápida para esse caso em específico!

Em resumo, efetivamente nós apenas jogamos fora os 4bits mais baixos de cada componente de cor no exemplo anterior.

Então podemos apenas aplicar uma operação de máscara de bits em cada componente de cor!

Repita o experimenro mas agora para 2bpp. Veja que agora não será mais necessária a etapa intermediária da imagem escura.

> pixel<sub>out</sub> = pixel<sub>in</sub> & (0b11000000, 0b11000000, 0b11000000)

In [ ]:
mask = np.array([0b11000000, 0b11000000, 0b11000000], dtype=np.uint8)
img_2bpp = img_8bpp & mask

ax = plt.subplot(1, 2, 1)
ax.imshow(img_8bpp); ax.axis('off'); ax.set_title('Original (8bpp)')

ax = plt.subplot(1, 2, 2)
ax.imshow(img_2bpp); ax.axis('off'); ax.set_title('Máscara AND (2bpp)')

plt.gcf().set_dpi(150)
plt.show()

### Exercício - Representação de Cores

Existem outras formas de representar as coreas nas imagens como o YUV e o HLS.

Utilizando cvtColor converta a imagem graf1.png, carregada colorida RGB para os formatos yuv e hsl, cada uma em sua prória variáevl.

Faça um plot 4x4, a primeira célula sendo a imagem rgb completa, e as próximas 3 colunas as componentes:

<table>
    <tr><td>graf</td><td>R</td><td>G</td><td>B</td></tr>
    <tr><td>&nbsp;</td><td>Y</td><td>U</td><td>V</td></tr>
    <tr><td>&nbsp;</td><td>H</td><td>L</td><td>S</td></tr>
</table>

Perceba que as componentes G e B são muito similares entre si, os tons cinza em possuem valore muito similares entre suas componentes.
Perceba que as componentes Y e o L são muito similares, pois ambos representam a luminosidade da cena.
Perceba que a componente S que representa a saturação está praticamente preta nos pontos onde a arte é cinza, indicando a falta de cor (H, hue)

> Utilize o colormap *'turbo'* para a componente H do HLS

In [ ]:
graf_bgr = cv.imread(cv.samples.findFile("graf1.png"))
graf_rgb = cv.cvtColor(graf_bgr, cv.COLOR_BGR2RGB)
graf_yuv = cv.cvtColor(graf_bgr, cv.COLOR_BGR2YUV)
graf_hls = cv.cvtColor(graf_bgr, cv.COLOR_BGR2HLS)

titles = [
    [('graf (RGB)', graf_rgb, None),  ('R', graf_rgb[:,:,0], 'gray'), ('G', graf_rgb[:,:,1], 'gray'),  ('B', graf_rgb[:,:,2], 'gray')],
    [(''         , None,      None),  ('Y', graf_yuv[:,:,0], 'gray'), ('U', graf_yuv[:,:,1], 'gray'),  ('V', graf_yuv[:,:,2], 'gray')],
    [(''         , None,      None),  ('H', graf_hls[:,:,0], 'turbo'),('L', graf_hls[:,:,1], 'gray'),  ('S', graf_hls[:,:,2], 'gray')],
]

fig, axes = plt.subplots(3, 4, figsize=(12, 8))
for row_i, row in enumerate(titles):
    for col_i, (title, data, cmap) in enumerate(row):
        ax = axes[row_i][col_i]
        if data is not None:
            ax.imshow(data, cmap=cmap)
            ax.set_title(title)
        else:
            ax.set_visible(False)
        ax.axis('off')
plt.tight_layout()
plt.show()

Muitas vezes, converter a imagem para o espaço de cores desejado ajuda na aplicação do filtro desejado.

Pegue o canal de cores (hue, H) e desloque os pixels para a direita 2 bits, de forma circular, faça um plot da imagem antes e depois do shit (preste atenção que você irá precisar converter a imagem HLS alterada para RGB para poder visualizar a mudança)

> Python não tem operador de rotação de bits, utilize o código

**Resultado esperado**: Os tons vermelhos irão ficar amarelados, os alaranjados irão ficar verdes, os tons azuis irão ficar róseos.

In [35]:
def rotl(n, d=2, width=8):
    d %= width
    return ((n << d) | (n >> (width - d))) & ((1 << width) - 1)

In [ ]:
hls_shifted = graf_hls.copy()
hls_shifted[:, :, 0] = np.vectorize(rotl)(graf_hls[:, :, 0])

graf_shifted_rgb = cv.cvtColor(hls_shifted, cv.COLOR_HLS2RGB)

plt.subplot(1, 2, 1)
plt.imshow(graf_rgb); plt.axis('off'); plt.title('Original RGB')

plt.subplot(1, 2, 2)
plt.imshow(graf_shifted_rgb); plt.axis('off'); plt.title('Hue rotacionado (rotl 2 bits)')

plt.gcf().set_dpi(150)
plt.show()